In [2]:
import os
os.chdir("./..")

In [11]:
import torch
from torch.utils.data import DataLoader
from pathlib import Path
import pandas as pd


from ml.data.tumor_dataset import TumorDataset
from ml.evaluation.evaluate_test import evaluate

In [8]:
path_test_images = "./data/splits/test/images" 
path_test_masks = "./data/splits/test/masks" 

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# One loader per preprocessing setup (see the check above)
test_loader_new = DataLoader(TumorDataset(path_test_images, path_test_masks, apply_transform=False),
                             batch_size=16, shuffle=False)
test_loader_old = DataLoader(TumorDataset(path_test_images, path_test_masks, apply_transform=False, normalize=False),
                             batch_size=16, shuffle=False)

results = {}
for setup, root, loader in [("baseline", "output", test_loader_old),
                            ("improved", "output/improved", test_loader_new)]:
    for m in ["unet", "unet++", "deeplabv3"]:
        results[(m, setup)] = evaluate(m, Path(root)/m/"best_model.pth", loader, device)

rows = [{"model": m, "setup": s,
         "test_dice": r["dice"].mean(), "dice_std": r["dice"].std(),
         "test_iou": r["iou"].mean(), "iou_std": r["iou"].std(),
         "params_M": r["params"]/1e6, "inference_ms": r["inference_ms"]}
        for (m, s), r in results.items()]
print(pd.DataFrame(rows).round(4))

       model     setup  test_dice  dice_std  test_iou  iou_std  params_M  \
0       unet  baseline     0.7746    0.2582    0.6865   0.2622   24.4301   
1     unet++  baseline     0.7704    0.2622    0.6827   0.2679   26.0723   
2  deeplabv3  baseline     0.7673    0.2554    0.6755   0.2604   26.0008   
3       unet  improved     0.7310    0.2783    0.6345   0.2710   24.4301   
4     unet++  improved     0.7650    0.2784    0.6800   0.2723   26.0723   
5  deeplabv3  improved     0.7828    0.2480    0.6937   0.2518   26.0008   

   inference_ms  
0     1257.1063  
1     2834.2342  
2     3887.1287  
3     1250.0868  
4     2801.8038  
5     3754.2519  


In [13]:
from scipy.stats import wilcoxon
import itertools

keys = list(results.keys())
for a, b in itertools.combinations(keys, 2):
    p = wilcoxon(results[a]["dice"], results[b]["dice"]).pvalue
    print(f"{a} vs {b}: p = {p:.4f}")

('unet', 'baseline') vs ('unet++', 'baseline'): p = 0.5661
('unet', 'baseline') vs ('deeplabv3', 'baseline'): p = 0.3820
('unet', 'baseline') vs ('unet', 'improved'): p = 0.0000
('unet', 'baseline') vs ('unet++', 'improved'): p = 0.4017
('unet', 'baseline') vs ('deeplabv3', 'improved'): p = 0.5261
('unet++', 'baseline') vs ('deeplabv3', 'baseline'): p = 0.3728
('unet++', 'baseline') vs ('unet', 'improved'): p = 0.0000
('unet++', 'baseline') vs ('unet++', 'improved'): p = 0.4275
('unet++', 'baseline') vs ('deeplabv3', 'improved'): p = 0.6482
('deeplabv3', 'baseline') vs ('unet', 'improved'): p = 0.0003
('deeplabv3', 'baseline') vs ('unet++', 'improved'): p = 0.2945
('deeplabv3', 'baseline') vs ('deeplabv3', 'improved'): p = 0.1732
('unet', 'improved') vs ('unet++', 'improved'): p = 0.0000
('unet', 'improved') vs ('deeplabv3', 'improved'): p = 0.0000
('unet++', 'improved') vs ('deeplabv3', 'improved'): p = 0.1083


---

DeepLabV3 (improved) obtains the highest test Dice (0.783), but its advantage over U-Net++ and over the baselines is not statistically significant (p = 0.108 and 0.173). Only U-Net (improved) is significantly worse than the others. The improved setup removed overfitting and instability but did not significantly raise test accuracy; since all improved models were still improving at epoch 25, longer training with early stopping may increase their performance and should be explored in future work.

---

1   -> 36   ==> (liste 1)
37  -> 76

            77  -> 100

101 -> 136  ==> (liste 2)
137 -> 176
177 -> 200

201 -> 236  ==> (liste3)
237 -> 276
277 -> 300

301 -> 336  ==> (liste 4)
337 -> 376

            377 -> 400

401 -> 436  ==> (liste 5)

            437 -> 476

477 -> 500